# DVF: Predict Next Mutation
# Author: Xavier VAN AUSLOOS, 9/02/26
# Notebook for preparing dataset (train/test) and training ML models for predicting next mutation


In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Get project root and ensure directories exist
_project_root = Path.cwd() if (Path.cwd() / "src" / "dvf").exists() else Path.cwd().parent
data_dir = _project_root / "data" / "processed"
models_dir = _project_root / "data" / "models"
results_dir = _project_root / "data" / "results"

for d in [data_dir, models_dir, results_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {data_dir}")
print(f"Models dir: {models_dir}")
print(f"Results dir: {results_dir}")


In [ ]:
# Load dataset from data/processed/
DATA_PATH = data_dir / "df_grouped_ensues_2020_2025_houses_cleaned.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
df.head()


In [ ]:
# Parse mutations JSON and extract features for next mutation prediction
def parse_mutations(mutations_str):
    """Parse mutations JSON string and return list of (date, value) tuples."""
    try:
        mutations = json.loads(mutations_str)
        parsed = []
        for mut in mutations:
            for date_str, value_str in mut.items():
                # Parse date
                date = pd.to_datetime(date_str, format="mixed", dayfirst=True, errors="coerce")
                # Parse value (handle comma decimal)
                value = pd.to_numeric(value_str.replace(",", "."), errors="coerce")
                if pd.notna(date) and pd.notna(value):
                    parsed.append((date, value))
        return sorted(parsed, key=lambda x: x[0])  # Sort by date
    except:
        return []

# Extract mutation features
df["mutations_parsed"] = df["mutations"].apply(parse_mutations)
df["n_mutations"] = df["mutations_parsed"].apply(len)

# Filter: need at least 2 mutations to predict next one
df_predictable = df[df["n_mutations"] >= 2].copy()
print(f"Houses with >= 2 mutations: {len(df_predictable)} / {len(df)}")


In [ ]:
# Create features and target for next mutation prediction
def extract_features(row):
    """Extract features from house and mutation history."""
    muts = row["mutations_parsed"]
    if len(muts) < 2:
        return None
    
    # Historical features
    dates = [m[0] for m in muts]
    values = [m[1] for m in muts]
    
    # Last mutation (will be used as target)
    last_date = dates[-1]
    last_value = values[-1]
    
    # Previous mutations (for features)
    prev_dates = dates[:-1]
    prev_values = values[:-1]
    
    # Time features
    days_since_first = (last_date - dates[0]).days if len(dates) > 1 else 0
    avg_days_between = np.mean([(dates[i+1] - dates[i]).days for i in range(len(dates)-1)]) if len(dates) > 1 else 0
    
    # Value features
    avg_value = np.mean(prev_values) if prev_values else 0
    value_trend = (last_value - prev_values[0]) / prev_values[0] if len(prev_values) > 0 and prev_values[0] > 0 else 0
    
    return {
        "n_prev_mutations": len(prev_values),
        "days_since_first": days_since_first,
        "avg_days_between": avg_days_between,
        "avg_prev_value": avg_value,
        "value_trend": value_trend,
        "last_value": last_value,
        "target_date": last_date,  # Next mutation date (using last as proxy)
        "target_value": last_value,  # Next mutation value
    }

# Extract features
features_list = []
for idx, row in df_predictable.iterrows():
    feat = extract_features(row)
    if feat:
        feat["idx"] = idx
        features_list.append(feat)

df_features = pd.DataFrame(features_list)
df_features = df_features.merge(df_predictable.reset_index(drop=True), left_on="idx", right_index=True, how="left")
print(f"Features extracted: {len(df_features)} rows")
df_features.head()


In [ ]:
# Prepare features (X) and targets (y)
# Features: house characteristics + mutation history features
feature_cols = [
    "Code postal",
    "Code commune",
    "Surface reelle bati",
    "Surface terrain",
    "Nombre pieces principales",
    "n_prev_mutations",
    "days_since_first",
    "avg_days_between",
    "avg_prev_value",
    "value_trend",
]

# Targets: next mutation date (days from reference) and value
X = df_features[feature_cols].copy()
X = X.fillna(0)  # Fill NaN

# Target 1: Days until next mutation (relative to last known date)
reference_date = df_features["target_date"].max()
y_days = (df_features["target_date"] - reference_date).dt.days

# Target 2: Next mutation value
y_value = df_features["target_value"]

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y_days.shape}, {y_value.shape}")


In [ ]:
# Split into train/test sets
X_train, X_test, y_days_train, y_days_test, y_value_train, y_value_test = train_test_split(
    X, y_days, y_value, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)} rows")
print(f"Test: {len(X_test)} rows")

# Save train/test sets
X_train.to_csv(data_dir / "X_train.csv", index=False)
X_test.to_csv(data_dir / "X_test.csv", index=False)
y_days_train.to_csv(data_dir / "y_days_train.csv", index=False)
y_days_test.to_csv(data_dir / "y_days_test.csv", index=False)
y_value_train.to_csv(data_dir / "y_value_train.csv", index=False)
y_value_test.to_csv(data_dir / "y_value_test.csv", index=False)

print(f"\nSaved train/test sets to {data_dir}")


In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model 1: Predict days until next mutation
model_days = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_days.fit(X_train_scaled, y_days_train)

# Train model 2: Predict next mutation value
model_value = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_value.fit(X_train_scaled, y_value_train)

print("Models trained successfully")


In [ ]:
# Save ML models
joblib.dump(model_days, models_dir / "model_predict_days.pkl")
joblib.dump(model_value, models_dir / "model_predict_value.pkl")
joblib.dump(scaler, models_dir / "scaler.pkl")

print(f"Saved models to {models_dir}")


In [ ]:
# Test ML models
# Predictions
y_days_pred = model_days.predict(X_test_scaled)
y_value_pred = model_value.predict(X_test_scaled)

# Metrics for days prediction
mae_days = mean_absolute_error(y_days_test, y_days_pred)
rmse_days = np.sqrt(mean_squared_error(y_days_test, y_days_pred))
r2_days = r2_score(y_days_test, y_days_pred)

# Metrics for value prediction
mae_value = mean_absolute_error(y_value_test, y_value_pred)
rmse_value = np.sqrt(mean_squared_error(y_value_test, y_value_pred))
r2_value = r2_score(y_value_test, y_value_pred)

print("=== Days Prediction Metrics ===")
print(f"MAE: {mae_days:.2f} days")
print(f"RMSE: {rmse_days:.2f} days")
print(f"R²: {r2_days:.4f}")

print("\n=== Value Prediction Metrics ===")
print(f"MAE: {mae_value:.2f} €")
print(f"RMSE: {rmse_value:.2f} €")
print(f"R²: {r2_value:.4f}")


In [ ]:
# Save test results
results = {
    "model_days": {
        "mae": float(mae_days),
        "rmse": float(rmse_days),
        "r2": float(r2_days),
    },
    "model_value": {
        "mae": float(mae_value),
        "rmse": float(rmse_value),
        "r2": float(r2_value),
    },
    "n_train": len(X_train),
    "n_test": len(X_test),
}

import json
with open(results_dir / "test_results.json", "w") as f:
    json.dump(results, f, indent=2)

# Save predictions
predictions_df = pd.DataFrame({
    "y_days_test": y_days_test.values,
    "y_days_pred": y_days_pred,
    "y_value_test": y_value_test.values,
    "y_value_pred": y_value_pred,
})
predictions_df.to_csv(results_dir / "predictions.csv", index=False)

print(f"\nSaved results to {results_dir}")
